![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [86]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

# Load the dataset
cc_apps = pd.read_csv("cc_approvals.data", header=None) 
cc_apps.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


## 1. Preprocessing the Data 

In [87]:
## check info 
cc_apps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       690 non-null    object 
 1   1       690 non-null    object 
 2   2       690 non-null    float64
 3   3       690 non-null    object 
 4   4       690 non-null    object 
 5   5       690 non-null    object 
 6   6       690 non-null    object 
 7   7       690 non-null    float64
 8   8       690 non-null    object 
 9   9       690 non-null    object 
 10  10      690 non-null    int64  
 11  11      690 non-null    object 
 12  12      690 non-null    int64  
 13  13      690 non-null    object 
dtypes: float64(2), int64(2), object(10)
memory usage: 75.6+ KB


In [88]:
# replace the np.NaN 
## cc_apps_clean = cc_apps.replace(to_replace=r"^[a-zA-A-Z]+$", value=np.nan, regex=True)
##cc_apps_clean = cc_apps_clean.replace("+", np.nan)
cc_apps_clean = cc_apps.replace(["?", "N/A", "null"],np.nan)
cc_apps_clean.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


In [89]:
# keep a copy 
cc_apps_clean_backup = cc_apps_clean.copy()

In [90]:
# impute value for 'object' column type 
object_cols = cc_apps_clean.select_dtypes(include=["object"]).columns
# fill the NaN with the most freq value 
cc_apps_clean[object_cols] = cc_apps_clean[object_cols].fillna(cc_apps_clean[object_cols].value_counts().iloc[0])
cc_apps_clean.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


In [91]:
# impute value for numeric column type 
num_cols = cc_apps_clean.select_dtypes(include=["number"]).columns
# impute numeric columns with mean 
cc_apps_clean[num_cols] = cc_apps_clean[num_cols].fillna(cc_apps_clean[num_cols].mean().iloc[0])
cc_apps_clean.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


In [92]:
# apply pd.get_dummy for One-hot encoding 
cc_apps_encoded = pd.get_dummies(cc_apps_clean, drop_first=True)
cc_apps_encoded.head()

,2,7,10,12,0_a,0_b,1_13.75,1_15.17,1_15.75,1_15.83,1_15.92,1_16.00,1_16.08,1_16.17,1_16.25,1_16.33,1_16.50,1_16.92,1_17.08,1_17.25,1_17.33,1_17.42,1_17.50,1_17.58,1_17.67,1_17.83,1_17.92,1_18.00,1_18.08,1_18.17,1_18.25,1_18.33,1_18.42,1_18.50,1_18.58,1_18.67,1_18.75,1_18.83,1_18.92,1_19.00,...,1_69.50,1_71.58,1_73.42,1_74.83,1_76.75,1_80.25,3_l,3_u,3_y,4_g,4_gg,4_p,5_aa,5_c,5_cc,5_d,5_e,5_ff,5_i,5_j,5_k,5_m,5_q,5_r,5_w,5_x,6_bb,6_dd,6_ff,6_h,6_j,6_n,6_o,6_v,6_z,8_t,9_t,11_p,11_s,13_-
0,0.000,1.25,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0
1,4.460,3.04,6,560,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0
2,0.500,1.50,0,824,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
3,1.540,3.75,5,3,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0
4,5.625,1.71,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0


## 2. Prepare data for Modeling 

In [93]:
# Extract the last column to be target variable 
X = cc_apps_encoded.iloc[:,:-1].values
y = cc_apps_encoded.iloc[:,[-1]].values

# Split into train_test set 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Initial StandardScaler and rescale X_train and X_test
scaler = StandardScaler()
rescale_Xtrain = scaler.fit_transform(X_train)
rescale_Xtest = scaler.transform(X_test)

#Initial Logistic Regression 
logreg = LogisticRegression()

# fit model 
logreg.fit(rescale_Xtrain,y_train)

#predict y 
y_pred = logreg.predict(rescale_Xtrain)

#print the confusion matrix of the logreg 
print(confusion_matrix(y_train,y_pred))


[[203   1]
 [  1 257]]


## 3. Evaluate the model 

In [94]:
# Defind the parameter of values for tolerance and max_iterations 
tol = [0.01,0.001,0.0001]
max_iter = [100,150,200]

# Create dict for pair tolerance and max_iterations 
param_grid = dict(tol=tol,max_iter=max_iter)

# Initial GridSearchCV 
grid_model = GridSearchCV(estimator=logreg,param_grid=param_grid,cv=5)

# Fit model to the data 
grid_model_result = grid_model.fit(rescale_Xtrain,y_train)

In [95]:
# Summarize 
best_train_score,best_train_params = grid_model_result.best_score_,grid_model_result.best_params_
print("Best: %f using %s" % (best_train_score,best_train_params))

Best: 0.813862 using {'max_iter': 100, 'tol': 0.01}


In [96]:
# Extract the best model and evaluate on the test data 
best_model = grid_model_result.best_estimator_
best_score = best_model.score(rescale_Xtest,y_test)

print("Accuracy of logistic regression classifier: ",best_score)

Accuracy of logistic regression classifier:  0.793859649122807
